# Cross-Model Shared Keyword SPAN Analysis

Compute SPAN for **ALL** keywords shared across ≥ 3 models.

**Output:**
- `shared_keyword_span.csv` — every shared keyword × every model
- `shared_keyword_span_summary.csv` — per-model avg-SPAN summary

In [1]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")

## Configuration

In [2]:
BASE = Path("/home/nedo/Kuliah/TA/Program")
DATA_DIR = BASE / "data" / "preprocess"
RESULTS_DIR = BASE / "results"

LIST_SUBJECT = ["cs", "math", "physics"]
MODELS = ["dtm", "lda", "top2vec", "bertopic", "topicGpt"]
MODEL_LABELS = {
    "dtm": "DTM", "lda": "LDA", "top2vec": "Top2Vec",
    "bertopic": "BERTopic", "topicGpt": "TopicGPT",
}

MIN_MODELS = 3        # keyword must appear in >= 3 models

for subject in LIST_SUBJECT:
    (RESULTS_DIR / "shared" / "tren" / subject).mkdir(parents=True, exist_ok=True)

print(f"Models: {list(MODEL_LABELS.values())}")
print(f"Min models for keyword inclusion: {MIN_MODELS}")
print(f"Subjects: {LIST_SUBJECT}")

Models: ['DTM', 'LDA', 'Top2Vec', 'BERTopic', 'TopicGPT']
Min models for keyword inclusion: 3
Subjects: ['cs', 'math', 'physics']


## Helper Functions

In [3]:
def compute_corpus_word_freq(subject):
    """Count each word across all documents in corpus (v̂ₖ)."""
    df = pd.read_csv(DATA_DIR / subject / "bow/v1.csv")
    word_freq = defaultdict(int)
    for text_val in df["text"]:
        try:
            tokens = ast.literal_eval(text_val)
            if isinstance(tokens, list):
                for w in tokens:
                    word_freq[w] += 1
        except (ValueError, SyntaxError):
            for w in str(text_val).split():
                word_freq[w] += 1
    return word_freq


def load_topic_words_by_year(model, subject):
    """Load topic-word evolution → {year: [set of words, ...]}."""
    evo_path = RESULTS_DIR / model / "temporal" / subject / "topic_word_evolution.csv"
    if not evo_path.exists():
        return {}, []
    evo_df = pd.read_csv(evo_path)
    years = sorted(evo_df["year"].unique())
    topic_words_by_year = defaultdict(list)
    for _, row in evo_df.iterrows():
        words = set(w.strip() for w in str(row["top_words"]).split(","))
        topic_words_by_year[int(row["year"])].append(words)
    return topic_words_by_year, years


def compute_span(keyword, topic_words_by_year, years):
    """SPAN = longest consecutive years keyword appears in any topic."""
    trend = []
    for y in years:
        found = any(keyword in words for words in topic_words_by_year.get(y, []))
        trend.append(1 if found else 0)
    max_span = 0
    current = 0
    for t in trend:
        if t == 1:
            current += 1
            max_span = max(max_span, current)
        else:
            current = 0
    return max_span, trend

---

# Part 1: Shared Keyword SPAN

Compute SPAN for **ALL** keywords shared across ≥ 3 models.

### 1.1 Collect Shared Terms (≥ 3 Models)

In [4]:
all_model_data = {}   # {subject: {model: (tw_by_year, years)}}
all_shared_keywords = {}  # {subject: [sorted list of words]}
all_corpus_freq = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Collecting terms: {subject.upper()}")
    print(f"{'='*70}")

    model_data = {}
    model_terms = defaultdict(set)

    for model in MODELS:
        tw, years = load_topic_words_by_year(model, subject)
        model_data[model] = (tw, years)
        for year_words in tw.values():
            for words_in_topic in year_words:
                model_terms[model].update(words_in_topic)
        print(f"  {MODEL_LABELS[model]:>10s}: {len(model_terms[model]):>6,} unique terms")

    # Count how many models discovered each term
    term_counts = defaultdict(int)
    for model, terms in model_terms.items():
        for term in terms:
            term_counts[term] += 1

    shared_terms = {term for term, count in term_counts.items() if count >= MIN_MODELS}
    print(f"\n  Terms in >= {MIN_MODELS} models: {len(shared_terms):,}")

    # Compute corpus frequency
    print(f"  Computing corpus word frequencies...")
    corpus_freq = compute_corpus_word_freq(subject)
    print(f"  Corpus vocabulary: {len(corpus_freq):,} unique words")

    # Sort by corpus frequency (descending)
    candidate_freqs = [(w, corpus_freq.get(w, 0)) for w in shared_terms]
    candidate_freqs.sort(key=lambda x: -x[1])
    shared_keywords = [w for w, _ in candidate_freqs]

    all_model_data[subject] = model_data
    all_shared_keywords[subject] = shared_keywords
    all_corpus_freq[subject] = corpus_freq


         DTM:    550 unique terms
         LDA:  2,616 unique terms
     Top2Vec: 10,251 unique terms
    BERTopic: 14,221 unique terms
    TopicGPT:  4,071 unique terms

  Terms in >= 3 models: 2,805
  Computing corpus word frequencies...
  Corpus vocabulary: 146,603 unique words

         DTM:    234 unique terms
         LDA:  2,090 unique terms
     Top2Vec:  7,194 unique terms
    BERTopic:  9,647 unique terms
    TopicGPT:  2,615 unique terms

  Terms in >= 3 models: 1,757
  Computing corpus word frequencies...
  Corpus vocabulary: 100,004 unique words

         DTM:    302 unique terms
         LDA:  2,046 unique terms
     Top2Vec:  8,166 unique terms
    BERTopic: 15,795 unique terms
    TopicGPT:  1,722 unique terms

  Terms in >= 3 models: 1,793
  Computing corpus word frequencies...
  Corpus vocabulary: 122,859 unique words


### 1.2 Compute SPAN for All Shared Keywords

In [5]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Shared Keyword SPAN: {subject.upper()}")
    print(f"{'='*70}")

    model_data = all_model_data[subject]
    corpus_freq = all_corpus_freq[subject]
    shared_keywords = all_shared_keywords[subject]
    total_shared = len(shared_keywords)
    ref_years = model_data["dtm"][1]

    print(f"  Shared keywords: {total_shared:,}")
    print(f"  Years: {ref_years[0]}–{ref_years[-1]} ({len(ref_years)} years)")

    # Compute SPAN for each keyword × each model
    rows = []
    for word in shared_keywords:
        v_hat = corpus_freq.get(word, 0)
        row = {"word": word, "v_hat": v_hat}

        for model in MODELS:
            tw, years = model_data[model]
            span, trend = compute_span(word, tw, years)
            s_dict = span / v_hat if v_hat > 0 else 0.0
            label = MODEL_LABELS[model]
            row[f"span_{label}"] = span
            row[f"trend_{label}"] = str(trend)
            row[f"s_dict_{label}"] = round(s_dict, 6)

        rows.append(row)

    result_df = pd.DataFrame(rows)

    # Save shared_keyword_span.csv
    out_dir = RESULTS_DIR / "shared" / "tren" / subject
    result_df.to_csv(out_dir / "shared_keyword_span.csv", index=False)

    # Summary: avg-SPAN per model
    summary_rows = []
    for model in MODELS:
        label = MODEL_LABELS[model]
        spans = result_df[f"span_{label}"]
        s_dicts = result_df[f"s_dict_{label}"]

        n_captured = int((spans > 0).sum())
        captured_mask = spans > 0

        avg_span_paper = s_dicts[captured_mask].mean() if n_captured > 0 else 0.0
        avg_span_simple = spans[captured_mask].mean() if n_captured > 0 else 0.0

        summary_rows.append({
            "model": label,
            "n_keywords": total_shared,
            "n_captured": n_captured,
            "capture_pct": round(n_captured / total_shared * 100, 2),
            "avg_span_paper": round(avg_span_paper, 8),
            "avg_span_simple": round(avg_span_simple, 4),
            "sum_s_dict": round(s_dicts.sum(), 6),
            "max_span": int(spans.max()),
            "n_full_span": int((spans == len(ref_years)).sum()),
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(out_dir / "shared_keyword_span_summary.csv", index=False)

    # Print results
    print(f"\n  {'Model':<10s} {'avg-SPAN(paper)':>16s} {'avg-SPAN(simple)':>16s} {'Captured':>12s} {'Full':>5s}")
    print(f"  {'-'*65}")
    for _, s in summary_df.iterrows():
        print(f"  {s['model']:<10s} {s['avg_span_paper']:>16.8f} {s['avg_span_simple']:>16.4f} "
              f"{s['n_captured']:>5d} ({s['capture_pct']:>5.1f}%) {s['n_full_span']:>5d}")

    # Top-10 keywords
    print(f"\n  Top 10 keywords (by corpus freq):")
    span_cols = [f"span_{MODEL_LABELS[m]}" for m in MODELS]
    print(f"  {'Word':<20s} {'v̂ₖ':>8s}  " + "  ".join(f"{MODEL_LABELS[m]:>8s}" for m in MODELS))
    for _, r in result_df.head(10).iterrows():
        spans_str = "  ".join(f"{r[c]:>8d}" for c in span_cols)
        print(f"  {r['word']:<20s} {r['v_hat']:>8d}  {spans_str}")

    print(f"\n  Saved: {out_dir / 'shared_keyword_span.csv'}")
    print(f"  Saved: {out_dir / 'shared_keyword_span_summary.csv'}")


Shared Keyword SPAN: CS
  Shared keywords: 2,805
  Years: 2000–2025 (26 years)

  Model       avg-SPAN(paper) avg-SPAN(simple)     Captured  Full
  -----------------------------------------------------------------
  DTM              0.00071500           6.5983   478 ( 17.0%)    17
  LDA              0.03449304           4.1433  1703 ( 60.7%)    33
  Top2Vec          0.04072019           5.5914  2780 ( 99.1%)    56
  BERTopic         0.02646373           3.3152  2646 ( 94.3%)     3
  TopicGPT         0.02946110           3.7547  2165 ( 77.2%)     6

  Top 10 keywords (by corpus freq):
  Word                      v̂ₖ       DTM       LDA   Top2Vec  BERTopic  TopicGPT
  datum                  116032        26        26        26         0        25
  algorithm               81182        26        26        26         4        26
  network                 80079        26        25        26        17        25
  image                   76073        17        23        23        16        1